# **Fine Tune with QLoRA**

Bu tuningda ROS2 uchun qilngan /content/sample_data/ros2_llama3_sft_large.jsonl  file bn train qilindi

In [ ]:
!python --version

Python 3.12.13


In [ ]:
!nvidia-smi

Thu Aug  6 03:31:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from  huggingface_hub import login
login(token = "your_hf_token")

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B",
                                             device_map="auto")


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [ ]:
tokenizer.add_special_tokens({"pad_token": "<|reserved_special_token_250|>"})
model.config.pad_token_id = tokenizer.pad_token_id
EOS_TOKEN = tokenizer.eos_token

tokenizer.apply_chat_template(chat, tokenize=False)

In [ ]:
!pip install -U datasets fsspec huggingface_hub pyarrow trl

  Using cached trl-1.9.2-py3-none-any.whl.metadata (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 25.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 17.0.0
    Uninstalling pyarrow-17.0.0:
      Successfully uninstalled pyarrow-17.0.0
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.6.1
    Uninstalling fsspec-2024.6.1:
      Successfully uninstalled fsspec-2024.6.1
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0
  Attempting uninstall: dat

In [ ]:
# config QLoRa

from peft import LoraConfig

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules = ["q_proj", "v_proj"]
    )


In [ ]:
from transformers import TrainingArguments
from trl import SFTConfig  # Bu TRL (Transformer Reinforcement Learning) kutubxonasiga tegishli

# training_arguments = TrainingArguments(
#     output_dir = "./llama3.2_1b_trained",
#     per_device_train_batch_size=3,
#     num_train_epochs=2,
#     learning_rate = 2e-4,
#     gradient_accumulation_steps=4,
#     tf32 = False,
#     fp16 = True,
#     logging_steps=10,
#     save_steps=100


# )


training_arguments = SFTConfig(
    output_dir="./llama3.2_1b_trained",

    num_train_epochs=1,
    per_device_train_batch_size=3,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=100,
    dataset_text_field="text"

    )

In [ ]:
from datasets import load_dataset    # real data bn ishlaydi
dataset = load_dataset('json', data_files="/content/sample_data/ros2_llama3_sft_large.jsonl", split = "train")

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
chat =

# **TEST MODEL BEFORE TRAIN  with validation data**

In [ ]:
print(dataset.column_names)

['messages', 'text', 'category']


In [ ]:
dataset = dataset.remove_columns(["messages", "category"])

print(dataset.column_names)

['text']


In [ ]:
import json
import random
import torch


model.eval()
# Load one sample from the val set
with open("/content/sample_data/ros2_llama3_sft_large_val.jsonl") as f:
    val_data = [json.loads(line) for line in f]

sample = random.choice(val_data)   # or val_data[0] for a fixed sample
category = sample.get("category", "n/a")

user_content = sample["messages"][1]["content"]      # instruction (+input)
reference_answer = sample["messages"][2]["content"]  # expected output

prompt = f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{user_content}

### Response:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)


with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.5,
        top_p=0.9,
        repetition_penalty=1.2,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )


result = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(f"=== Category: {category} ===\n")
print(result)
print("\n" + "=" * 80)
print("REFERENCE ANSWER:\n")
print(reference_answer)

=== Category: control ===

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Produce the Nav2 NavigateToPose action goal to send the robot to x=0.82, y=4.67 in the map frame, facing -90 degrees (yaw).

### Response:
```
navx = navx.NavX()
pose_goal = PoseGoal(x=0.82,y=4.67)
move_base.set_pose_goal(pose_goal)

# Wait for move base's execution of this command
while not rospy.is_shutdown():
    pass

print("Done!")
```

REFERENCE ANSWER:

```yaml
pose:
  header:
    frame_id: 'map'
  pose:
    position:
      x: 0.82
      y: 4.67
      z: 0.0
    orientation:
      x: 0.0
      y: 0.0
      z: -0.707
      w: 0.707
```
Sent to the /navigate_to_pose action server (nav2_msgs/action/NavigateToPose).


In [ ]:
# Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported so uninstall it for no
!pip uninstall -y torchao

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=training_arguments,


)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/3126 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3126 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/3126 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3126 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/3126 [00:00<?, ? examples/s]

In [ ]:
trainer.train()


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Step,Training Loss
10,2.482805
20,1.845534
30,1.230895
40,0.921449
50,0.745065
60,0.638632
70,0.615426
80,0.570915
90,0.549253
100,0.510275


TrainOutput(global_step=261, training_loss=0.6894162603265024, metrics={'train_runtime': 610.8718, 'train_samples_per_second': 5.117, 'train_steps_per_second': 0.427, 'total_flos': 4884703614996480.0, 'train_loss': 0.6894162603265024, 'entropy': 0.5512907803058624, 'num_tokens': 708491.0, 'mean_token_accuracy': 0.9123773574829102, 'epoch': 1.0})

In [ ]:
trainer.save_model("./myLlama3.2")

In [ ]:
trainer.model.print_trainable_parameters()

trainable params: 851,968 || all params: 1,236,666,368 || trainable%: 0.0689


In [ ]:
model.eval()

import torch

model.eval()

prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Produce the Nav2 NavigateToPose action goal to send the robot to x=0.82, y=4.67 in the map frame, facing -90 degrees (yaw).

### Response:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)


with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.5,
        top_p=0.9,
        repetition_penalty=1.2,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )


result = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(f"=== Category: {category} ===\n")
print(result)
print("\n" + "=" * 80)
print("REFERENCE ANSWER:\n")
print(reference_answer)

=== Category: control ===

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Produce the Nav2 NavigateToPose action goal to send the robot to x=0.82, y=4.67 in the map frame, facing -90 degrees (yaw).

### Response:
```json
{
  "type": "navigate_to_pose",
  "pose": {
    "header": {},
    "position": [
      0.082,
      4.667
    ],
    "orientation": [
      -1.5708,
      0.00000,
      0.8660,
      0.00000
    ]
  },
  "action_group_id": null,
  "group_name": ""
}
```


REFERENCE ANSWER:

```yaml
pose:
  header:
    frame_id: 'map'
  pose:
    position:
      x: 0.82
      y: 4.67
      z: 0.0
    orientation:
      x: 0.0
      y: 0.0
      z: -0.707
      w: 0.707
```
Sent to the /navigate_to_pose action server (nav2_msgs/action/NavigateToPose).
